# 01_02 - Exogenous Variables: Weather Data Extraction
## 1. Methodology Overview

To analyze energy market resilience, our **Operational Shield** requires high-fidelity meteorological context to anticipate fluctuations in the Spot market. 

**Extraction Architecture (Batch Processing):**
The meteorological data was sourced from the **Open-Meteo Archive API**, querying daily metrics for 52 distinct Spanish provinces from January 2020 onwards. 

Because requesting thousands of data points across 52 locations simultaneously can trigger API timeouts and rate-limiting, the extraction was systematically divided into batches. This logic is modularized in the external script `src/data/extract_weather.py`.

The script processes the geographic coordinates provided in `_Provincias_Info.xlsx`, loops through the API requests, and stores the raw responses natively.

### 1.2. The Raw Data Artifacts
Executing the script generated multiple localized batches representing the raw API outputs:
* `daily_top0-10_cities_..._extracted.csv`
* `daily_top11-51_cities_..._extracted.csv`

In this notebook, we simply load a sample of these locally stored batches to verify the structural integrity of the extraction before moving to the EDA and Aggregation pipelines.

In [1]:
import pandas as pd
from pathlib import Path
import sys
import os

sys.path.append(os.path.abspath('../../src'))

raw_weather_dir = Path("../../data/raw/weather")

batch_1_path = list(raw_weather_dir.glob("*top0-10*.csv"))[0]
batch_2_path = list(raw_weather_dir.glob("*top11-51*.csv"))[0]

print(f"Found Batch 1: {batch_1_path.name}")
print(f"Found Batch 2: {batch_2_path.name}")

# Load the first batch to inspect the raw extraction
df_raw_sample = pd.read_csv(batch_1_path)

print(f"\n✅ Raw Weather Data Sample Loaded Successfully.")
print(f"Total rows in sample: {len(df_raw_sample)}")

Found Batch 1: daily_top0-10_cities_2020-2026_2026-04-05_extracted.csv
Found Batch 2: daily_top11-51_cities_2020-2026_2026-04-04_extracted.csv

✅ Raw Weather Data Sample Loaded Successfully.
Total rows in sample: 25025


## 2. Visualizing the Raw Extraction

Let's inspect the raw format of the API response. At this stage, the data is entirely untransformed—it exists as a pure time series of meteorological variables (temperature, radiation, wind, etc.) strictly segmented by individual cities.

In [2]:
# Display a sample of the raw meteorological features
display_cols = ['date', 'city', 'temperature_2m_mean', 'wind_speed_10m_max', 'shortwave_radiation_sum']
display(df_raw_sample[display_cols].head())

# Verify the cities included in this specific batch
cities_extracted = df_raw_sample['city'].unique()
print(f"\nCities encompassed in this batch ({len(cities_extracted)} total):")
print(", ".join(cities_extracted))

,date,city,temperature_2m_mean,wind_speed_10m_max,shortwave_radiation_sum
0,2020-01-01 00:00:00+00:00,Madrid,3.147917,9.178235,9.33
1,2020-01-02 00:00:00+00:00,Madrid,2.483333,8.534353,9.23
2,2020-01-03 00:00:00+00:00,Madrid,1.568750,8.209263,6.72
3,2020-01-04 00:00:00+00:00,Madrid,3.754167,17.057314,8.87
4,2020-01-05 00:00:00+00:00,Madrid,3.179167,9.931042,9.41



Cities encompassed in this batch (11 total):
Madrid, Barcelona, Valencia, Alicante, Sevilla, Málaga, Murcia, Cádiz, Islas Baleares, Las Palmas, Vizcaya


## 3. Pipeline Execution: National Aggregation

We now execute the `process_weather.py` script. This engine performs the following operations in memory:
1. Concatenates the raw regional batches.
2. Merges the dataset with `Provinces_data.csv` to acquire demographic and geographic weights.
3. Applies the Dual-Weighting mathematical transformation.
4. Engineers advanced Peninsular extremes (filtering out islands to capture systemic Iberian shocks).
5. Exports the consolidated dataset to the `interim` data layer.

In [3]:
import sys
import os
from pathlib import Path

# Dynamically configure the project root directory
project_root = Path(os.path.abspath('../../'))
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Re-importing the function (now correctly loaded after restart)
from src.data.process_weather import aggregate_weather_batches

# Define operational paths
raw_weather_dir = project_root / "data" / "raw" / "weather"
provinces_info_path = project_root / "data" / "external" / "Provinces_data.csv"
output_aggregated_path = project_root / "data" / "interim" / "national_weather_aggregated.csv"

aggregate_weather_batches(
    raw_weather_dir=raw_weather_dir,
    provinces_info_file=provinces_info_path,
    output_file=output_aggregated_path
)

Step 1: Loading raw meteorological batches...
Step 2: Processing weights from Provinces_data.csv...
Step 3: Applying Dual-Weighting Strategy...
Step 4: Engineering advanced features...
✅ SUCCESS! File generated: c:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\interim\national_weather_aggregated.csv


## 4. Extraction Verification

To confirm the aggregation engine ran successfully, we load the resulting `national_weather_aggregated.csv` and inspect its unified structure.

In [6]:
# Load the newly created aggregated dataset
df_weather_agg = pd.read_csv(output_aggregated_path)
df_weather_agg['date'] = pd.to_datetime(df_weather_agg['date'])

print(f"✅ National Aggregated Weather Dataset Loaded.")
print(f"Temporal Scope: {df_weather_agg['date'].min().date()} to {df_weather_agg['date'].max().date()}")
print(f"Dimensions: {df_weather_agg.shape[0]} days x {df_weather_agg.shape[1]} features\n")

# Display a sample demonstrating the transition from 'Provincial' to 'National Weighted' data
# CORRECCIÓN: Cambiado 'peninsular_max_temperature' por 'peninsular_max_temp'
display_cols = ['date', 'temperature_2m_mean', 'wind_speed_10m_max', 'peninsular_max_temp']
display(df_weather_agg[display_cols].head())

✅ National Aggregated Weather Dataset Loaded.
Temporal Scope: 2020-01-01 to 2026-03-24
Dimensions: 2275 days x 21 features



,date,temperature_2m_mean,wind_speed_10m_max,peninsular_max_temp
0,2020-01-01,6.535541,9.113214,16.05
1,2020-01-02,6.054612,9.889774,15.10
2,2020-01-03,6.174448,11.881113,15.15
3,2020-01-04,6.525313,13.375087,14.65
4,2020-01-05,6.535853,11.201827,15.75
